# Preprocessing Teks

In [ ]:
# %pip install pandas nltk Sastrawi

## Library

In [ ]:
import re
import html
import polars as pl
import nltk
# import spacy
import unicodedata
from functools import lru_cache

# import pandas as pd

from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
nltk.download("stopwords")

# Model spaCy untuk bahasa Indonesia
# spacy.cli.download("xx_ent_wiki_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Musa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Helper Fungsion

In [2]:
def to_list(v):
    """map_elements memberi pl.Series untuk kolom list -> ubah ke list Python."""
    if v is None: return []
    if isinstance(v, pl.Series): return v.to_list()
    return list(v)
 
def _apply_str(df, col, out, fn):
    return df.with_columns(pl.col(col).map_elements(fn, return_dtype=pl.Utf8).alias(out))
 
def _apply_list(df, col, out, fn):
    return df.with_columns(
        pl.col(col).map_elements(lambda v: fn(to_list(v)), return_dtype=pl.List(pl.Utf8)).alias(out))

## Dataset

In [4]:
df = pl.read_csv("../politik.csv")

In [5]:
df

type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,lang,bookmarkCount,isReply,inReplyToId,conversationId,inReplyToUserId,inReplyToUsername,isPinned,author,extendedEntities,card,place,entities,reply_to_user_results,quoted_tweet_results,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex,isQuote
str,i64,str,str,str,str,i64,i64,i64,i64,i64,str,str,i64,bool,i64,i64,i64,str,bool,str,str,str,str,str,str,str,str,str,bool,i64,bool
"""tweet""",2108447840284668323,"""https://x.com/Jezaelans/status…","""https://twitter.com/Jezaelans/…","""@MenteriPekerja Mknya dr awal …",null,0,0,0,0,1,"""Fri Oct 09 06:41:52 +0000 2026""","""in""",0,true,2108337734675017861,2108337734675017861,1717188745227862016,"""MenteriPekerja""",false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'user_mentions': [{'id_str': …","""{'rest_id': '17171887452278620…",null,null,null,false,0,null
"""tweet""",2108446988073402729,"""https://x.com/rang_simabua/sta…","""https://twitter.com/rang_simab…","""Pernyataan Rocky Gerung yang m…",null,2,0,2,0,30,"""Fri Oct 09 06:38:29 +0000 2026""","""in""",0,false,null,2108446988073402729,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'hashtags': [], 'symbols': []…",null,"""{'rest_id': '21081447174480077…","""{'type': 'tweet', 'id': '21081…",null,false,0,true
"""tweet""",2108446836512141677,"""https://x.com/BiLLRaY2019/stat…","""https://twitter.com/BiLLRaY201…","""Wantimpres angkat bicara..! R…",null,0,0,0,0,46,"""Fri Oct 09 06:37:53 +0000 2026""","""in""",0,false,null,2108446836512141677,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'additional_media_…",null,"""{}""","""{}""",null,null,null,null,false,0,null
"""tweet""",2108445787856458122,"""https://x.com/triwahyuutama15/…","""https://twitter.com/triwahyuut…","""Pembentukan Wantimpres oleh Pr…",null,0,0,0,0,5,"""Fri Oct 09 06:33:43 +0000 2026""","""in""",0,false,null,2108445787856458122,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,null,null,null,false,0,null
"""tweet""",2108445386075721812,"""https://x.com/Putra_Riau11/sta…","""https://twitter.com/Putra_Riau…","""Ternyata 1 orang seperti anda …",null,0,0,0,0,4,"""Fri Oct 09 06:32:07 +0000 2026""","""in""",0,false,null,2108445386075721812,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'additional_media_…",null,"""{}""","""{}""",null,null,null,null,false,0,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tweet""",2108179303100002598,"""https://x.com/kodam_siliwangi/…","""https://twitter.com/kodam_sili…","""Kasdam III/Siliwangi Brigjen T…",null,0,0,0,0,98,"""Thu Oct 08 12:54:48 +0000 2026""","""in""",0,false,null,2108179303100002598,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'allow_download_st…",null,"""{}""","""{}""",null,null,null,null,false,1,null
"""tweet""",2108178863297859718,"""https://x.com/kkpgoid/status/2…","""https://twitter.com/kkpgoid/st…","""Menteri Trenggono @saktitrengg…",null,0,0,0,0,111,"""Thu Oct 08 12:53:03 +0000 2026""","""in""",0,true,2108178427111211244,2108178427111211244,2493957026,"""kkpgoid""",false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'hashtags': [{'indices': [220…","""{'rest_id': '2493957026', 'res…",null,null,null,false,1,null
"""tweet""",2108178461231874433,"""https://x.com/BpsipSulut/statu…","""https://twitter.com/BpsipSulut…","""BRMP Sulut Dampingi Koperasi U…",null,0,0,0,0,13,"""Thu Oct 08 12:51:27 +0000 2026""","""in""",0,false,null,2108178461231874433,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'display_url': 'pi…",null,"""{}""","""{'urls': [{'display_url': 'fac…",null,null,null,null,false,1,null


In [6]:
df = df.select("text")
df

text
str
"""@MenteriPekerja Mknya dr awal …"
"""Pernyataan Rocky Gerung yang m…"
"""Wantimpres angkat bicara..! R…"
"""Pembentukan Wantimpres oleh Pr…"
"""Ternyata 1 orang seperti anda …"
…
"""Kasdam III/Siliwangi Brigjen T…"
"""Menteri Trenggono @saktitrengg…"
"""BRMP Sulut Dampingi Koperasi U…"


In [7]:
df.dtypes

[String]

## Noise Removal

Noise Removal digunakan untuk membersihkan atau menghapus elemen yang biasanya tidak dibutuhkan dalam analisis teks, seperti HTML, URL, alamat email, mention, karakter kontrol, dan spasi berlebih yang tidak relevan dan tidak memiliki nilai informasi untuk analisis teks.

Contoh yang dibersihkan: Tanda baca, emoji, tag HTML, URL, karakter khusus (seperti @, #, $), dan spasi berlebih (whitespace).

### Code

In [ ]:
EMOJI = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF\uFE0F\u200d]")

In [ ]:
def noise_removal(df, text_col, output_col="text_clean"):
    def clean(text):
        if text is None: return ""
        text = html.unescape(str(text))
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+|\b\w+\.(?:co|com|id)/\S+", " ", text)
        text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)
        text = re.sub(r"(?<!\w)@\w+", " ", text)
        text = re.sub(r"^\s*RT\b", " ", text)
        # hashtag: buang '#', pisahkan CamelCase
        text = re.sub(r"#(\w+)", lambda m: " " + re.sub(r"(?<=[a-z])(?=[A-Z])", " ", m.group(1)) + " ", text)
        text = EMOJI.sub(" ", text)
        text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, clean)

In [9]:
df_noise = noise_removal(
    df, text_col="text"
)

In [10]:
df_noise

text,text_clean
str,str
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …"
…,…
"""Kasdam III/Siliwangi Brigjen T…","""Kasdam III/Siliwangi Brigjen T…"
"""Menteri Trenggono @saktitrengg…","""Menteri Trenggono berharap fas…"
"""BRMP Sulut Dampingi Koperasi U…","""BRMP Sulut Dampingi Koperasi U…"


In [11]:
print(type(df))


<class 'polars.dataframe.frame.DataFrame'>


## Normalisasi Teks

Normalisasi teks adalah proses menyeragamkan format teks, terutama huruf kapital, Unicode, dan spasi agar memiliki struktur linguistik yang seragam, terutama untuk teks tidak formal, bahasa gaul, singkatan, atau typo (typhographical errors).

Contoh: Mengubah kata "bgt" menjadi "banget", "udh" menjadi "sudah", atau menstandarisasi penulisan angka dan simbol mata uang.

### Code

In [ ]:
SLANG = {
    "yg": "yang", "dg": "dengan", "dgn": "dengan", "utk": "untuk", "krn": "karena",
    "karna": "karena", "tdk": "tidak", "gak": "tidak", "ga": "tidak", "nggak": "tidak",
    "gk": "tidak", "enggak": "tidak", "ngga": "tidak", "bkn": "bukan", "bkan": "bukan",
    "sdh": "sudah", "udh": "sudah", "udah": "sudah", "blm": "belum", "jg": "juga",
    "jga": "juga", "sm": "sama", "sma": "sama", "klo": "kalau", "kalo": "kalau",
    "klu": "kalau", "trus": "terus", "tp": "tapi", "tpi": "tapi", "aja": "saja",
    "bgt": "banget", "emg": "memang", "emang": "memang", "org": "orang", "ornag": "orang",
    "gw": "saya", "gue": "saya", "gua": "saya", "lu": "kamu", "loe": "kamu",
    "dr": "dari", "dlm": "dalam", "pd": "pada", "sbg": "sebagai", "spt": "seperti",
    "mknya": "makanya", "pkiran": "pikiran", "stlah": "setelah", "mnjbat": "menjabat",
    "ko": "kok", "mentri": "menteri", "thn": "tahun", "nglantik": "melantik",
    "latarbelakang": "latar belakang", "ehh": "eh", "wajarlah": "wajar lah",
    "rg": "rocky gerung"
}

In [12]:
def text_normalization(df, text_col, output_col="text_normalized", slang=SLANG):
    def norm(text):
        if not text: return ""
        text = unicodedata.normalize("NFKC", text).lower()
        text = re.sub(r"[“”„‟]", '"', text)
        text = re.sub(r"[‘’‚‛]", "'", text)
        text = re.sub(r"[—–]", " - ", text)
        text = re.sub(r"(\w)\1{2,}", r"\1", text)                  # mantaaap -> mantap
        text = re.sub(r"\b([a-z]+)2\b", r"\1-\1", text)             # tiba2 -> tiba-tiba
        text = re.sub(r"\b[a-z0-9]+\b", lambda m: slang.get(m.group(0), m.group(0)), text)
        return re.sub(r"\s+", " ", text).strip()
    return _apply_str(df, text_col, output_col, norm)

In [13]:
df_normalized = text_normalization(
    df_noise, text_col="text_clean"
)

In [14]:
df_normalized

text,text_clean,text_normalized
str,str,str
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …","""makanya dari awal kok aneh pra…"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…","""pernyataan rocky gerung yang m…"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…","""wantimpres angkat bicara..! ro…"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…","""pembentukan wantimpres oleh pr…"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …"
…,…,…
"""Kasdam III/Siliwangi Brigjen T…","""Kasdam III/Siliwangi Brigjen T…","""kasdam i/siliwangi brigjen tni…"
"""Menteri Trenggono @saktitrengg…","""Menteri Trenggono berharap fas…","""menteri trenggono berharap fas…"
"""BRMP Sulut Dampingi Koperasi U…","""BRMP Sulut Dampingi Koperasi U…","""brmp sulut dampingi koperasi u…"


## Tokenisasi

Memecah kalimat atau paragraf yang panjang menjadi unit-unit kecil yang disebut token (biasanya berupa kata atau frasa), hasilnya berupa list, bukan string..

Contoh: Kalimat "Saya sedang belajar NLP." dipecah menjadi: ["Saya", "sedang", "belajar", "NLP"].

### Code

In [ ]:
# def tokenization(df, text_col, output_col="tokens"):
#     """
#     Memecah teks menjadi token kata.
#     Output setiap baris berupa list.
#     """
#     result = df.clone()

#     def tokenize_text(text):
#         if text is None:
#             return ""
#         # if pl.isna(text):
#         #     return []

#         text = str(text)

#         # Ambil rangkaian karakter alfanumerik.
#         # Kata dengan apostrof atau tanda hubung dipisah.
#         return re.findall(r"\b\w+\b", text, flags=re.UNICODE)

#     return df.with_columns(
#         df[text_col].map_elements(tokenize_text).alias(output_col)
#     )
#     # result[output_col] = result[text_col].apply(tokenize_text)
#     # return result

In [15]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")
def tokenization(df, text_col, output_col="tokens"):
    return df.with_columns(
        pl.col(text_col).map_elements(lambda t: TOKEN_RE.findall(t or ""),
                                      return_dtype=pl.List(pl.Utf8)).alias(output_col))
 

In [16]:
df_tokens = tokenization(
    df_normalized, text_col="text_normalized"
)

In [17]:
df_tokens

text,text_clean,text_normalized,tokens
str,str,str,list[str]
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …","""makanya dari awal kok aneh pra…","[""makanya"", ""dari"", … ""gila""]"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…","""pernyataan rocky gerung yang m…","[""pernyataan"", ""rocky"", … ""diinginkan""]"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…","""wantimpres angkat bicara..! ro…","[""wantimpres"", ""angkat"", … ""massa""]"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…","""pembentukan wantimpres oleh pr…","[""pembentukan"", ""wantimpres"", … ""pelantikan""]"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …","[""ternyata"", ""1"", … ""wantimpres""]"
…,…,…,…
"""Kasdam III/Siliwangi Brigjen T…","""Kasdam III/Siliwangi Brigjen T…","""kasdam i/siliwangi brigjen tni…","[""kasdam"", ""i"", … ""cimahi""]"
"""Menteri Trenggono @saktitrengg…","""Menteri Trenggono berharap fas…","""menteri trenggono berharap fas…","[""menteri"", ""trenggono"", … ""trenggono""]"
"""BRMP Sulut Dampingi Koperasi U…","""BRMP Sulut Dampingi Koperasi U…","""brmp sulut dampingi koperasi u…","[""brmp"", ""sulut"", … ""selengkapnya""]"


## Stopwords Removal

Menghapus kata-kata umum yang sering muncul dalam suatu bahasa namun memiliki sedikit atau bahkan tidak memiliki makna analitik yang kuat.

Contoh: dan, yang, di, dari, ke, untuk

### Code

In [18]:
# Stopword bahasa Indonesia
stop_words = set(stopwords.words("indonesian"))

# Tambahan stopword opsional
custom_stopwords = {
    "yg", "dg", "dgn", "nya", "sih", "dong",
    "nih", "deh", "aja", "udah", "gak", "nggak"
}

stop_words.update(custom_stopwords)

In [19]:
def build_stopwords(keep_negation=True):
    sw = set(stopwords.words("indonesian"))
    sw |= {"sih", "dong", "nih", "deh", "kok", "lah", "kan", "nya", "tuh", "nah", "yah", "ya"}
    if keep_negation:
        sw -= {"tidak", "bukan", "tak", "belum", "jangan", "tanpa"}
    return sw
 
def stopword_removal(df, text_col, output_col="tokens_no_stopwords", stop_words=None):
    stop_words = stop_words or build_stopwords()
    return _apply_list(df, text_col, output_col,
                       lambda toks: [t for t in toks if t not in stop_words and len(t) > 1])

In [20]:
df_no_stopwords = stopword_removal(
    df_tokens, text_col="tokens"
)

In [21]:
df_no_stopwords

text,text_clean,text_normalized,tokens,tokens_no_stopwords
str,str,str,list[str],list[str]
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …","""makanya dari awal kok aneh pra…","[""makanya"", ""dari"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…","""pernyataan rocky gerung yang m…","[""pernyataan"", ""rocky"", … ""diinginkan""]","[""pernyataan"", ""rocky"", … ""tak""]"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…","""wantimpres angkat bicara..! ro…","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…","""pembentukan wantimpres oleh pr…","[""pembentukan"", ""wantimpres"", … ""pelantikan""]","[""pembentukan"", ""wantimpres"", … ""pelantikan""]"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …","[""ternyata"", ""1"", … ""wantimpres""]","[""orang"", ""berharga"", … ""wantimpres""]"
…,…,…,…,…
"""Kasdam III/Siliwangi Brigjen T…","""Kasdam III/Siliwangi Brigjen T…","""kasdam i/siliwangi brigjen tni…","[""kasdam"", ""i"", … ""cimahi""]","[""kasdam"", ""siliwangi"", … ""cimahi""]"
"""Menteri Trenggono @saktitrengg…","""Menteri Trenggono berharap fas…","""menteri trenggono berharap fas…","[""menteri"", ""trenggono"", … ""trenggono""]","[""menteri"", ""trenggono"", … ""trenggono""]"
"""BRMP Sulut Dampingi Koperasi U…","""BRMP Sulut Dampingi Koperasi U…","""brmp sulut dampingi koperasi u…","[""brmp"", ""sulut"", … ""selengkapnya""]","[""brmp"", ""sulut"", … ""selengkapnya""]"


In [22]:
df_no_stopwords['tokens_no_stopwords'][0]

""
str
"""aneh"""
"""prabowo"""
"""tidak"""
"""melantik"""
"""wantimpres"""
…
"""menjabat"""
"""pelantikan"""
"""26"""


## Stemming

Proses memotong imbuhan (prefiks, sufiks, infiks) pada sebuah kata untuk dikembalikan ke bentuk dasarnya (stem), tanpa selalu memperhatikan kaidah kamus yang valid.

Contoh: Kata "memakan", "dimakan", dan "termakan" akan diubah menjadi akar kata dasar "makan".

### Code

In [23]:
# Inisialisasi stemmer bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()


In [ ]:
# def stemming(df: pl.DataFrame, text_col: str, output_col: str = "tokens_stemmed"):
#     """
#     Melakukan stemming menggunakan Sastrawi.
#     Menerima teks atau list token.
#     """
#     def stem_text(value):
#         if value is None:
#             return []  # selalu list
#         elif isinstance(value, list):
#             tokens = value
#         else:
#             tokens = str(value).split()

#         return [stemmer.stem(str(token)) for token in tokens]

#     return df.with_columns(
#         df[text_col].map_elements(
#             stem_text,
#             return_dtype=pl.List(pl.Utf8)
#         ).alias(output_col)
#     )


In [ ]:
STEM_EXCEPTIONS = {"belasan": "belas"}
PROTECTED = {"wantimpres", "gibran", "prabowo", "rocky", "gerung", "pratikno", "dpr"}

In [26]:
_stemmer = StemmerFactory().create_stemmer()
@lru_cache(maxsize=None)
def _stem(tok):
    if tok in PROTECTED or tok.isdigit() or "-" in tok: return tok
    if tok in STEM_EXCEPTIONS: return STEM_EXCEPTIONS[tok]
    return _stemmer.stem(tok)
 
def stemming(df, text_col, output_col="tokens_stemmed"):
    return _apply_list(df, text_col, output_col, lambda toks: [_stem(t) for t in toks])

In [27]:
df_stemmed = stemming(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [28]:
df_stemmed

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed
str,str,str,list[str],list[str],list[str]
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …","""makanya dari awal kok aneh pra…","[""makanya"", ""dari"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…","""pernyataan rocky gerung yang m…","[""pernyataan"", ""rocky"", … ""diinginkan""]","[""pernyataan"", ""rocky"", … ""tak""]","[""nyata"", ""rocky"", … ""tak""]"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…","""wantimpres angkat bicara..! ro…","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…","""pembentukan wantimpres oleh pr…","[""pembentukan"", ""wantimpres"", … ""pelantikan""]","[""pembentukan"", ""wantimpres"", … ""pelantikan""]","[""bentuk"", ""wantimpres"", … ""lantik""]"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …","[""ternyata"", ""1"", … ""wantimpres""]","[""orang"", ""berharga"", … ""wantimpres""]","[""orang"", ""harga"", … ""wantimpres""]"
…,…,…,…,…,…
"""Kasdam III/Siliwangi Brigjen T…","""Kasdam III/Siliwangi Brigjen T…","""kasdam i/siliwangi brigjen tni…","[""kasdam"", ""i"", … ""cimahi""]","[""kasdam"", ""siliwangi"", … ""cimahi""]","[""kasdam"", ""siliwangi"", … ""cimahi""]"
"""Menteri Trenggono @saktitrengg…","""Menteri Trenggono berharap fas…","""menteri trenggono berharap fas…","[""menteri"", ""trenggono"", … ""trenggono""]","[""menteri"", ""trenggono"", … ""trenggono""]","[""menteri"", ""trenggono"", … ""trenggono""]"
"""BRMP Sulut Dampingi Koperasi U…","""BRMP Sulut Dampingi Koperasi U…","""brmp sulut dampingi koperasi u…","[""brmp"", ""sulut"", … ""selengkapnya""]","[""brmp"", ""sulut"", … ""selengkapnya""]","[""brmp"", ""sulut"", … ""lengkap""]"


In [29]:
df_stemmed['tokens_stemmed'][0]


""
str
"""aneh"""
"""prabowo"""
"""tidak"""
"""lantik"""
"""wantimpres"""
…
"""jabat"""
"""lantik"""
"""26"""


## Lemmatization

Proses mengubah kata berimbuhan menjadi bentuk dasarnya berdasarkan kamus dan tata bahasa yang valid (lemma), dengan memperhatikan konteks kalimat. Berbeda dengan stemming, proses ini idealnya menghasilkan bentuk lema yang valid secara linguistik.

### Code

In [44]:
def lemmatization(
    df: pl.DataFrame,
    text_col: str,
    output_col: str = "tokens_lemmatized",
    lemma_dict: dict = None
):
    """
    Lemmatization berbasis kamus.
    Kata yang tidak ada dalam kamus dipertahankan.
    """
    if lemma_dict is None:
        lemma_dict = lemma_dictionary

    def lemmatize_text(value):
        if value is None:
            return []
        # kolom list di Polars masuk sebagai pl.Series -> ubah ke list Python
        if isinstance(value, pl.Series):
            tokens = value.to_list()
        elif isinstance(value, list):
            tokens = value
        else:
            tokens = str(value).split()

        return [lemma_dict.get(str(t).lower(), str(t).lower()) for t in tokens]

    return df.with_columns(
        pl.col(text_col).map_elements(
            lemmatize_text,
            return_dtype=pl.List(pl.Utf8)
        ).alias(output_col)
    )

In [45]:
df_lemmatized = lemmatization(
    df_no_stopwords, text_col="tokens_no_stopwords"
)

In [46]:
df_lemmatized

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_lemmatized
str,str,str,list[str],list[str],list[str]
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …","""makanya dari awal kok aneh pra…","[""makanya"", ""dari"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…","""pernyataan rocky gerung yang m…","[""pernyataan"", ""rocky"", … ""diinginkan""]","[""pernyataan"", ""rocky"", … ""tak""]","[""pernyataan"", ""rocky"", … ""tak""]"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…","""wantimpres angkat bicara..! ro…","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…","""pembentukan wantimpres oleh pr…","[""pembentukan"", ""wantimpres"", … ""pelantikan""]","[""pembentukan"", ""wantimpres"", … ""pelantikan""]","[""pembentukan"", ""wantimpres"", … ""pelantikan""]"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …","[""ternyata"", ""1"", … ""wantimpres""]","[""orang"", ""berharga"", … ""wantimpres""]","[""orang"", ""berharga"", … ""wantimpres""]"
…,…,…,…,…,…
"""Kasdam III/Siliwangi Brigjen T…","""Kasdam III/Siliwangi Brigjen T…","""kasdam i/siliwangi brigjen tni…","[""kasdam"", ""i"", … ""cimahi""]","[""kasdam"", ""siliwangi"", … ""cimahi""]","[""kasdam"", ""siliwangi"", … ""cimahi""]"
"""Menteri Trenggono @saktitrengg…","""Menteri Trenggono berharap fas…","""menteri trenggono berharap fas…","[""menteri"", ""trenggono"", … ""trenggono""]","[""menteri"", ""trenggono"", … ""trenggono""]","[""menteri"", ""trenggono"", … ""trenggono""]"
"""BRMP Sulut Dampingi Koperasi U…","""BRMP Sulut Dampingi Koperasi U…","""brmp sulut dampingi koperasi u…","[""brmp"", ""sulut"", … ""selengkapnya""]","[""brmp"", ""sulut"", … ""selengkapnya""]","[""brmp"", ""sulut"", … ""selengkapnya""]"


## Hasil

In [47]:
columns_to_view = [
    "text",
    "text_clean",
    "text_normalized",
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

df_comparison = (
    df_lemmatized
    .with_columns(df_stemmed["tokens_stemmed"])
    .select(columns_to_view)
)

df_comparison.head()

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,list[str],list[str],list[str],list[str]
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …","""makanya dari awal kok aneh pra…","[""makanya"", ""dari"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]","[""aneh"", ""prabowo"", … ""gila""]"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…","""pernyataan rocky gerung yang m…","[""pernyataan"", ""rocky"", … ""diinginkan""]","[""pernyataan"", ""rocky"", … ""tak""]","[""nyata"", ""rocky"", … ""tak""]","[""pernyataan"", ""rocky"", … ""tak""]"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…","""wantimpres angkat bicara..! ro…","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]","[""wantimpres"", ""angkat"", … ""massa""]"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…","""pembentukan wantimpres oleh pr…","[""pembentukan"", ""wantimpres"", … ""pelantikan""]","[""pembentukan"", ""wantimpres"", … ""pelantikan""]","[""bentuk"", ""wantimpres"", … ""lantik""]","[""pembentukan"", ""wantimpres"", … ""pelantikan""]"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …","[""ternyata"", ""1"", … ""wantimpres""]","[""orang"", ""berharga"", … ""wantimpres""]","[""orang"", ""harga"", … ""wantimpres""]","[""orang"", ""berharga"", … ""wantimpres""]"


In [48]:
df_export = df_comparison.clone()

token_columns = [
    "tokens",
    "tokens_no_stopwords",
    "tokens_stemmed",
    "tokens_lemmatized"
]

# Ubah list token menjadi string
for col in token_columns:
    df_export = df_export.with_columns(
        df_export[col].map_elements(
            lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens),
            return_dtype=pl.Utf8
        ).alias(col)
    )

# # Simpan hasil ke CSV
# df_export.write_csv(
#     "hasil_preprocessing.csv",
#     encoding="utf8"   # Polars tidak pakai "utf-8-sig", cukup "utf8"
# )


In [49]:
df_export = df_comparison.with_columns(
    [pl.col(c).list.join(" ") for c in token_columns]
)

df_export.write_csv("hasil_preprocessing2.csv")

In [50]:
df_export.write_csv("hasil_preprocessing2.csv")


In [51]:
df_load = pl.read_csv('hasil_preprocessing2.csv')

In [52]:
df_load

text,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_stemmed,tokens_lemmatized
str,str,str,str,str,str,str
"""@MenteriPekerja Mknya dr awal …","""Mknya dr awal ko aneh Prabowo …","""makanya dari awal kok aneh pra…","""makanya dari awal kok aneh pra…","""aneh prabowo tidak melantik wa…","""aneh prabowo tidak lantik want…","""aneh prabowo tidak melantik wa…"
"""Pernyataan Rocky Gerung yang m…","""Pernyataan Rocky Gerung yang m…","""pernyataan rocky gerung yang m…","""pernyataan rocky gerung yang m…","""pernyataan rocky gerung wapres…","""nyata rocky gerung wapres mund…","""pernyataan rocky gerung wapres…"
"""Wantimpres angkat bicara..! R…","""Wantimpres angkat bicara..! Ro…","""wantimpres angkat bicara..! ro…","""wantimpres angkat bicara rocky…","""wantimpres angkat bicara rocky…","""wantimpres angkat bicara rocky…","""wantimpres angkat bicara rocky…"
"""Pembentukan Wantimpres oleh Pr…","""Pembentukan Wantimpres oleh Pr…","""pembentukan wantimpres oleh pr…","""pembentukan wantimpres oleh pr…","""pembentukan wantimpres prabowo…","""bentuk wantimpres prabowo lang…","""pembentukan wantimpres prabowo…"
"""Ternyata 1 orang seperti anda …","""Ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …","""ternyata 1 orang seperti anda …","""orang berharga puluhan menteri…","""orang harga puluh menteri bela…","""orang berharga puluhan menteri…"
…,…,…,…,…,…,…
"""Kasdam III/Siliwangi Brigjen T…","""Kasdam III/Siliwangi Brigjen T…","""kasdam i/siliwangi brigjen tni…","""kasdam i siliwangi brigjen tni…","""kasdam siliwangi brigjen tni a…","""kasdam siliwangi brigjen tni a…","""kasdam siliwangi brigjen tni a…"
"""Menteri Trenggono @saktitrengg…","""Menteri Trenggono berharap fas…","""menteri trenggono berharap fas…","""menteri trenggono berharap fas…","""menteri trenggono berharap fas…","""menteri trenggono harap fasili…","""menteri trenggono berharap fas…"
"""BRMP Sulut Dampingi Koperasi U…","""BRMP Sulut Dampingi Koperasi U…","""brmp sulut dampingi koperasi u…","""brmp sulut dampingi koperasi u…","""brmp sulut dampingi koperasi u…","""brmp sulut damping koperasi us…","""brmp sulut dampingi koperasi u…"
